In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Data ingestion

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

In [0]:
display(df)

#Silver transformations

In [0]:
#Trim extra spaces

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
#Separating prd_key and cat_key

df = df.withColumn('cat_key', regexp_replace(substring(col('prd_key'), 1, 5), '-', '_'))
df = df.withColumn('prd_key', substring(col('prd_key'), 7, length(col('prd_key'))))

In [0]:
cols = df.columns
cols.remove("cat_key")

df = df.select(
    *cols[:2],
    "cat_key",
    *cols[2:]
)

In [0]:
display(df)

In [0]:
#Cost Cleanup

df = df.withColumn('prd_cost', coalesce(col('prd_cost'), lit(0)))

In [0]:
#Standardizations for "prd_line"

df = df.withColumn('prd_line',
    when(upper(col('prd_line')) == 'M', 'Mountain')
    .when(upper(col('prd_line')) == 'S', 'Other Sales')
    .when(upper(col('prd_line')) == 'R', 'Road')
    .when(upper(col('prd_line')) == 'T', 'Touring')
    .otherwise('n/a')
    )

In [0]:
#rename columns

RENAME_COL = {
    'prd_id': 'product_id',
    'prd_key': 'product_key',
    'cat_key': 'category_key',
    'prd_nm': 'product_name',
    'prd_cost': 'product_cost',
    'prd_line': 'product_line',
    'prd_start_dt': 'product_start_date',
    'prd_end_dt': 'product_end_date'
}

for old, new in RENAME_COL.items():
    df = df.withColumnRenamed(old ,new)

#Writing Silver Table

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver.crm_products')

In [0]:
%sql
select * from workspace.silver.crm_products limit 5